# View_History + Movie

In [50]:
import pandas as pd

vh_0 = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_promotion_split_v1\promotion_0_view_history_v2.csv"
)

vh_1 = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_promotion_split_v1\promotion_1_view_history_v2.csv"
)

movie = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Movie_Master_v2.csv"
)

print("vh_0 길이:", len(vh_0))
print("vh_1 길이:", len(vh_1))
print("movie 길이:", len(movie))
print("movie 종류 개수:", movie["MOVIE_NUM"].nunique())

vh_0 길이: 82824
vh_1 길이: 88705
movie 길이: 14502
movie 종류 개수: 14018


영화 중복 제거

In [51]:
import pandas as pd

movie = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Movie_Master_v2.csv"
)

key_col = "MOVIE_NUM"
compare_cols = [col for col in movie.columns if col != key_col]

# MOVIE_NUM별 등장 횟수 계산부
movie_num_count = movie.groupby(key_col)[key_col].transform("size")

# MOVIE_NUM이 2번 이상 나온 행 판별부
is_duplicate_key = movie_num_count.ge(2)

# MOVIE_NUM과 나머지 컬럼까지 모두 동일한 중복행 판별부
is_same_row_duplicate = movie.duplicated(
    subset=[key_col] + compare_cols,
    keep="first"
)

# 삭제 대상 판별부
drop_mask = is_duplicate_key & is_same_row_duplicate

# 정리 결과 생성부
movie_clean = movie.loc[~drop_mask].copy()

print("원본 행 수:", len(movie))
print("삭제 대상 행 수:", int(drop_mask.sum()))
print("정리 후 행 수:", len(movie_clean))
print("정리 후 movie 종류 수:", movie_clean["MOVIE_NUM"].nunique())

원본 행 수: 14502
삭제 대상 행 수: 475
정리 후 행 수: 14027
정리 후 movie 종류 수: 14018


In [52]:
# 정리 후에도 2번 이상 등장하는 MOVIE_NUM 추출부
remaining_duplicate_counts = (
    movie_clean["MOVIE_NUM"]
    .value_counts()
    .loc[lambda s: s >= 2]
    .sort_index()
)

print("정리 후에도 2번 이상 등장하는 MOVIE_NUM 개수:", len(remaining_duplicate_counts))
print(remaining_duplicate_counts)

# 해당 MOVIE_NUM의 상세 행 확인부
remaining_duplicate_rows = (
    movie_clean[movie_clean["MOVIE_NUM"].isin(remaining_duplicate_counts.index)]
    .sort_values(["MOVIE_NUM"])
    .copy()
)

print("\n정리 후에도 중복인 행들")
print(remaining_duplicate_rows)


정리 후에도 2번 이상 등장하는 MOVIE_NUM 개수: 9
MOVIE_NUM
1435     2
2012     2
3967     2
4946     2
5763     2
6475     2
9664     2
10508    2
12135    2
Name: count, dtype: int64

정리 후에도 중복인 행들
       MOVIE_NUM movie_title  ott_release_month             genre
1434        1435  아나콘다(1997)             202003  Action/Adventure
1435        1435  아나콘다(1997)             202003            Comedy
2002        2012          뮬란             201609  Action/Adventure
2003        2012          뮬란             201609  Animation/Family
3959        3967    뮬란(2020)             202010  Action/Adventure
3960        3967    뮬란(2020)             202010  Animation/Family
4929        4946     드래곤길들이기             201211  Animation/Family
4931        4946     드래곤길들이기             201211  Action/Adventure
5747        5763  인어공주(1989)             201912        SF/Fantasy
5746        5763  인어공주(1989)             201912           Romance
6458        6475         메리미             201610            Comedy
6459        6475        

In [53]:
# 정리 후에도 2번 이상 등장하는 MOVIE_NUM 키 추출부
remaining_duplicate_keys = remaining_duplicate_counts.index

# 잔여 중복 키의 후행 행 제거부
keep_mask = ~(
    movie_clean["MOVIE_NUM"].isin(remaining_duplicate_keys) &
    movie_clean.duplicated(subset=["MOVIE_NUM"], keep="first")
)

# 최종 정리 결과 생성부
movie_final = movie_clean.loc[keep_mask].copy()

print("중간 정리 후 행 수:", len(movie_clean))
print("최종 정리 후 행 수:", len(movie_final))
print("최종 movie 종류 수:", movie_final["MOVIE_NUM"].nunique())

# 최종 중복 여부 확인부
final_duplicate_counts = (
    movie_final["MOVIE_NUM"]
    .value_counts()
    .loc[lambda s: s >= 2]
    .sort_index()
)

print("최종 정리 후에도 2번 이상 등장하는 MOVIE_NUM 개수:", len(final_duplicate_counts))
print(final_duplicate_counts)

중간 정리 후 행 수: 14027
최종 정리 후 행 수: 14018
최종 movie 종류 수: 14018
최종 정리 후에도 2번 이상 등장하는 MOVIE_NUM 개수: 0
Series([], Name: count, dtype: int64)


In [54]:
movie_final.to_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\data\01_initial\movie_final.csv",
    index=False,
    encoding="utf-8-sig"
)

In [55]:
vh_0_merged = vh_0.merge(
    movie_final,
    on="MOVIE_NUM",
    how="left"
)

vh_1_merged = vh_1.merge(
    movie_final,
    on="MOVIE_NUM",
    how="left"
)

In [56]:
print("vh_0_merged 길이:", len(vh_0_merged))
print("vh_0_merged의 고유 USER_NUM 개수:", vh_0_merged["USER_NUM"].nunique())
print("vh_1_merged 길이:", len(vh_1_merged))
print("vh_1_merged의 고유 USER_NUM 개수:", vh_1_merged["USER_NUM"].nunique())

vh_0_merged 길이: 82824
vh_0_merged의 고유 USER_NUM 개수: 11261
vh_1_merged 길이: 88705
vh_1_merged의 고유 USER_NUM 개수: 11957


# Membership + User_mapping

In [57]:
import pandas as pd

membership_0 = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_promotion_split_v1\promotion_0_membership_v2.csv"
)

mapping_0 = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_promotion_split_v1\promotion_0_user_mapping_v2.csv"
)


membership_1 = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_promotion_split_v1\promotion_1_membership_v2.csv"
)

mapping_1 = pd.read_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_promotion_split_v1\promotion_1_user_mapping_v2.csv"
)

In [58]:
all_userkey_in_membership = mapping_0["USER_KEY"].isin(membership_0["USER_KEY"]).all()
print("mapping_0의 USER_KEY가 membership_0에 전부 존재하는지 여부:", all_userkey_in_membership)
all_userkey_in_membership = mapping_1["USER_KEY"].isin(membership_1["USER_KEY"]).all()
print("mapping_1의 USER_KEY가 membership_1에 전부 존재하는지 여부:", all_userkey_in_membership)

mapping_0의 USER_KEY가 membership_0에 전부 존재하는지 여부: True
mapping_1의 USER_KEY가 membership_1에 전부 존재하는지 여부: True


In [59]:
membership_0_temp = membership_0.copy()
mapping_0_temp = mapping_0.copy()

membership_0_temp["merge_order"] = membership_0_temp.groupby("USER_KEY").cumcount()
mapping_0_temp["merge_order"] = mapping_0_temp.groupby("USER_KEY").cumcount()

membership_mapping_0 = mapping_0_temp.merge(
    membership_0_temp,
    on=["USER_KEY", "merge_order"],
    how="left"
)

membership_mapping_0 = membership_mapping_0.drop(columns=["merge_order"])

print("병합 결과 행 개수:", len(membership_mapping_0))
print("병합 결과 고유 USER_NUM 개수:", membership_mapping_0["USER_NUM"].nunique())
print("병합 결과 고유 USER_KEY 개수:", membership_mapping_0["USER_KEY"].nunique())

병합 결과 행 개수: 11261
병합 결과 고유 USER_NUM 개수: 11261
병합 결과 고유 USER_KEY 개수: 11221


In [60]:
membership_1_temp = membership_1.copy()
mapping_1_temp = mapping_1.copy()

membership_1_temp["merge_order"] = membership_1_temp.groupby("USER_KEY").cumcount()
mapping_1_temp["merge_order"] = mapping_1_temp.groupby("USER_KEY").cumcount()

membership_mapping_1 = mapping_1_temp.merge(
    membership_1_temp,
    on=["USER_KEY", "merge_order"],
    how="left"
)

membership_mapping_1 = membership_mapping_1.drop(columns=["merge_order"])

print("병합 결과 행 개수:", len(membership_mapping_1))
print("병합 결과 고유 USER_NUM 개수:", membership_mapping_1["USER_NUM"].nunique())
print("병합 결과 고유 USER_KEY 개수:", membership_mapping_1["USER_KEY"].nunique())


병합 결과 행 개수: 11957
병합 결과 고유 USER_NUM 개수: 11957
병합 결과 고유 USER_KEY 개수: 11951


# 최종 병합

In [61]:
vh_membership_0_merged = vh_0_merged.merge(
    membership_mapping_0,
    on="USER_NUM",
    how="left"
)

print("병합 결과 행 개수:", len(vh_membership_0_merged))
print("vh_0_merged 행 개수:", len(vh_0_merged))
print("병합 결과 고유 USER_NUM 개수:", vh_membership_0_merged["USER_NUM"].nunique())
print("병합 결과 고유 USER_KEY 개수:", vh_membership_0_merged["USER_KEY"].nunique())


병합 결과 행 개수: 82824
vh_0_merged 행 개수: 82824
병합 결과 고유 USER_NUM 개수: 11261
병합 결과 고유 USER_KEY 개수: 11221


In [62]:
vh_membership_1_merged = vh_1_merged.merge(
    membership_mapping_1,
    on="USER_NUM",
    how="left"
)

print("병합 결과 행 개수:", len(vh_membership_1_merged))
print("vh_1_merged 행 개수:", len(vh_1_merged))
print("병합 결과 고유 USER_NUM 개수:", vh_membership_1_merged["USER_NUM"].nunique())
print("병합 결과 고유 USER_KEY 개수:", vh_membership_1_merged["USER_KEY"].nunique())


병합 결과 행 개수: 88705
vh_1_merged 행 개수: 88705
병합 결과 고유 USER_NUM 개수: 11957
병합 결과 고유 USER_KEY 개수: 11951


In [63]:
vh_membership_0_merged.to_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\data\260509_merged1_0.csv",
    index=False,
    encoding="utf-8-sig"
)

vh_membership_1_merged.to_csv(
    r"C:\myCode\ott-churn-prediction\kim.kwangil\data\260509_merged1_1.csv",
    index=False,
    encoding="utf-8-sig"
)